# EDA · Kenya · perinatal + ANC detail

Dataset: `data/processed/ke_perinatal_anc.csv`.
Outcome: `perinatal_death` (stillbirth or death in the first 7 days).

Adds the ANC predictors: anc_visits, first_anc_month, bp_measured, urine_tested, blood_tested, iron_tablets.

What this notebook should cover:
- shape and dtypes; confirm the row count matches the proposal
- missingness (expect blanks in told_high_bp, preceding_interval_m, and the ANC columns)
- the outcome rate
- distributions of the predictors (maternal_age, prior_pregnancies, prior_live_births, prior_stillbirth, prior_loss, preceding_interval_m, multiple, education, wealth_quintile, residence, region, told_high_bp) and the outcome rate within each category

Clear outputs before committing.

## 1. Data Loading and Initial Inspection

This section loads the processed Kenya perinatal and antenatal care dataset and examines its structure, dimensions, variable types, and completeness.

The objective is to establish a clear understanding of the dataset before undertaking data cleaning. No records will be removed or values modified during this initial inspection.

In [ ]:
# Import the required libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Locate the repository root
current_path = Path.cwd()

possible_roots = [
    current_path,
    *current_path.parents
]

repo_root = next(
    (
        path for path in possible_roots
        if (path / "contracts" / "cohort_config.yaml").exists()
    ),
    None
)

if repo_root is None:
    raise FileNotFoundError(
        "Repository root not found. "
        "Ensure the notebook is running inside the Perinatal-risk repository."
    )

# Define the dataset path
data_path = repo_root / "data" / "processed" / "ke_perinatal_anc.csv"

# Load the dataset
df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print(f"Dataset location: {data_path}")

In [ ]:
# Check dataset dimensions
rows, columns = df.shape

print(f"Number of rows: {rows:,}")
print(f"Number of columns: {columns:,}")

In [ ]:
# Display all column names
for number, column in enumerate(df.columns, start=1):
    print(f"{number:02}. {column}")

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
# Prediction targets
TARGET_PERINATAL = "perinatal_death"
TARGET_LBW = "low_birth_weight"

# Predictors common to all datasets
CORE_PREDICTORS = [
    "maternal_age",
    "education",
    "wealth_quintile",
    "residence",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "multiple"
]

# Additional predictors for ANC and LBW datasets
ANC_PREDICTORS = [
    "anc_visits",
    "first_anc_month",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets"
]

# Helper columns: not model predictors
HELPER_COLUMNS = [
    "weight",
    "v001"
]

# EDA-only columns: not model predictors
EDA_ONLY_COLUMNS = [
    "region",
    "told_high_bp"
]

# Fields explicitly excluded from model inputs
POST_OUTCOME_COLUMNS = [
    "pregnancy_months",
    "delivery_place",
    "baby_sex",
    "birth_weight_g"
]

print("AfyaNova feature specification loaded.")
print(f"Core predictors: {len(CORE_PREDICTORS)}")
print(f"Additional ANC/LBW predictors: {len(ANC_PREDICTORS)}")

In [ ]:
# Combine all expected columns for the Kenya ANC dataset

expected_columns = (
    [TARGET_PERINATAL]
    + CORE_PREDICTORS
    + ANC_PREDICTORS
    + HELPER_COLUMNS
    + EDA_ONLY_COLUMNS
    + POST_OUTCOME_COLUMNS
)

# Identify missing and available columns
missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

available_columns = [
    col for col in expected_columns
    if col in df.columns
]

print(f"Expected columns checked: {len(expected_columns)}")
print(f"Available columns: {len(available_columns)}")
print(f"Missing columns: {len(missing_columns)}")

if missing_columns:
    print("\nColumns requiring investigation:")
    for col in missing_columns:
        print(f"- {col}")
else:
    print("\nAll expected columns are present.")

## Data Quality Assessment

This section assesses the Kenyan perinatal and antenatal care dataset before modelling. The checks cover data types, missing values, duplicate records, unique identifiers, and the distribution of the target variable.

The original dataset will remain unchanged during this assessment. Any cleaning decisions will be guided by the findings and the agreed Group 13 feature specification.

In [ ]:
# Inspect data types and non-null counts
df.info()

In [ ]:
# Calculate missing values and percentages
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2),
    "data_type": df.dtypes.astype(str)
})

# Show columns with missing values first
missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

display(missing_summary)

In [ ]:
# Count exact duplicate rows
exact_duplicates = df.duplicated().sum()

print(f"Total records: {len(df):,}")
print(f"Exact duplicate rows: {exact_duplicates:,}")

In [ ]:
# Examine the perinatal death outcome
print("Target variable: perinatal_death")
print("\nCounts:")
display(df["perinatal_death"].value_counts(dropna=False).sort_index())

print("\nPercentages:")
display(
    (df["perinatal_death"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
    .sort_index()
)

In [ ]:
# Check whether caseid and pidx are available
identifier_columns = ["caseid", "pidx"]

for column in identifier_columns:
    if column in df.columns:
        print(f"{column}:")
        print(f"  Missing values: {df[column].isna().sum():,}")
        print(f"  Unique values: {df[column].nunique(dropna=True):,}")
        print(f"  Duplicate values: {df[column].duplicated().sum():,}")

In [ ]:
# Check whether caseid + pidx uniquely identifies each record

print("Unique caseid + pidx combinations:",
      df[["caseid", "pidx"]].drop_duplicates().shape[0])

print("Total records:", len(df))

print("Duplicate caseid + pidx combinations:",
      df.duplicated(subset=["caseid", "pidx"]).sum())

In [ ]:
# Identify case IDs that occur more than once
caseid_counts = df["caseid"].value_counts()

repeated_caseids = caseid_counts[caseid_counts > 1]

print("Number of repeated caseids:", len(repeated_caseids))
print("Records belonging to repeated caseids:",
      repeated_caseids.sum())

# Display a few examples
repeated_ids = repeated_caseids.index[:5]

display(
    df[df["caseid"].isin(repeated_ids)]
    .sort_values(["caseid", "pidx"])
)

In [ ]:
# Validate the target's basic integrity

print("Missing target values:", df["perinatal_death"].isna().sum())

print("Observed target values:",
      sorted(df["perinatal_death"].dropna().unique()))

print("\nTarget distribution:")
display(
    df["perinatal_death"]
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# Examine preceding birth intervals by previous live births

# Verify the column names and data types
print(df[["prior_live_births", "preceding_interval_m"]].dtypes)

# Display the frequency distribution of prior live births
print("\nPrior live births distribution:")
display(df["prior_live_births"].value_counts(dropna=False).sort_index())

# Recalculate missing preceding intervals
interval_check = (
    df.groupby("prior_live_births", dropna=False)
      .agg(
          records=("preceding_interval_m", "size"),
          missing_intervals=("preceding_interval_m",
                             lambda x: x.isna().sum())
      )
      .reset_index()
)

interval_check["missing_percentage"] = (
    interval_check["missing_intervals"]
    .div(interval_check["records"].replace(0, np.nan))
    .mul(100)
    .round(2)
)

display(interval_check)

In [ ]:
# Inspect the repeated case IDs
caseid_counts = df["caseid"].value_counts()
repeated_ids = caseid_counts[caseid_counts > 1].index

# Compare repeated records
repeated_records = (
    df[df["caseid"].isin(repeated_ids)]
    .sort_values(["caseid", "pidx"])
)

display(
    repeated_records[
        [
            "caseid",
            "pidx",
            "prior_live_births",
            "prior_stillbirth",
            "prior_loss",
            "preceding_interval_m",
            "perinatal_death"
        ]
    ].head(20)
)

In [ ]:
# Inspect variables that may help validate the outcome
outcome_candidates = [
    col for col in [
        "perinatal_death",
        "p4",
        "p6",
        "stillbirth",
        "early_neonatal_death"
    ]
    if col in df.columns
]

print("Available outcome-related columns:")
print(outcome_candidates)

for col in outcome_candidates:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).sort_index().head(20))

In [ ]:
# Inspect the two components of the perinatal-death outcome
for col in ["stillbirth", "early_neonatal_death"]:
    print(f"\n{col} distribution:")
    display(df[col].value_counts(dropna=False).sort_index())

# Check for missing component values
print("\nMissing stillbirth values:", df["stillbirth"].isna().sum())
print(
    "Missing early neonatal death values:",
    df["early_neonatal_death"].isna().sum()
)

# Reconstruct the target, provided both components are complete
if df[["stillbirth", "early_neonatal_death"]].notna().all().all():

    reconstructed_target = (
        (df["stillbirth"] == 1)
        | (df["early_neonatal_death"] == 1)
    ).astype(int)

    mismatches = (
        df["perinatal_death"] != reconstructed_target
    ).sum()

    print("\nTarget validation results")
    print("Original positive outcomes:", int(df["perinatal_death"].sum()))
    print("Reconstructed positive outcomes:", int(reconstructed_target.sum()))
    print("Mismatched records:", int(mismatches))

else:
    print(
        "\nSome component values are missing. "
        "Investigate them before validating the reconstructed target."
    )

In [ ]:
# Recalculate the missing-interval analysis
history_check = (
    df.groupby(
        ["prior_live_births", "prior_stillbirth", "prior_loss"],
        dropna=False
    )
    .agg(
        records=("caseid", "size"),
        missing_intervals=("preceding_interval_m", lambda x: x.isna().sum())
    )
    .reset_index()
)

history_check["missing_percentage"] = (
    history_check["missing_intervals"]
    .div(history_check["records"])
    .mul(100)
    .round(2)
)

# Keep groups with missing intervals
history_missing = history_check[
    history_check["missing_intervals"] > 0
].sort_values("missing_intervals", ascending=False)

display(history_missing.head(20))

In [ ]:
print("Total records:", len(df))
print("Total missing preceding intervals:",
      df["preceding_interval_m"].isna().sum())

print("Records across history groups:",
      history_check["records"].sum())

print("Missing intervals across history groups:",
      history_check["missing_intervals"].sum())

In [ ]:
# Inspect the underlying pregnancy-history variables
history_columns = [
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss"
]

for col in history_columns:
    print(f"\n{col}")
    print("Data type:", df[col].dtype)
    print("Missing values:", df[col].isna().sum())
    print("Unique values:", sorted(df[col].dropna().unique()))
    print(df[col].value_counts(dropna=False).sort_index())

In [ ]:
# Inspect records with missing preceding intervals
missing_interval_records = df.loc[
    df["preceding_interval_m"].isna(),
    [
        "caseid",
        "pidx",
        "prior_live_births",
        "prior_stillbirth",
        "prior_loss",
        "preceding_interval_m"
    ]
]

display(missing_interval_records.head(15))

In [ ]:
# Count records with no previous live births, stillbirths, or losses
no_previous_history = (
    (df["prior_live_births"] == 0)
    & (df["prior_stillbirth"] == 0)
    & (df["prior_loss"] == 0)
)

print("Records with all three history values equal to zero:",
      int(no_previous_history.sum()))

print("Missing intervals in this group:",
      int(df.loc[no_previous_history, "preceding_interval_m"].isna().sum()))

print("Non-missing intervals in this group:",
      int(df.loc[no_previous_history, "preceding_interval_m"].notna().sum()))

In [ ]:
# Identify missing intervals outside the no-previous-history group

no_previous_history = (
    (df["prior_live_births"] == 0)
    & (df["prior_stillbirth"] == 0)
    & (df["prior_loss"] == 0)
)

other_missing_intervals = df.loc[
    df["preceding_interval_m"].isna() & ~no_previous_history,
    [
        "caseid",
        "pidx",
        "prior_live_births",
        "prior_stillbirth",
        "prior_loss",
        "preceding_interval_m"
    ]
]

print("Other records with missing intervals:",
      len(other_missing_intervals))

display(other_missing_intervals.head(50))

In [ ]:
# agreed predictor specification

CORE_PREDICTORS = [
    "maternal_age",
    "education",
    "wealth_quintile",
    "residence",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "multiple"
]

ANC_PREDICTORS = [
    "anc_visits",
    "first_anc_month",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets"
]

PREDICTORS = CORE_PREDICTORS + ANC_PREDICTORS

# Check that all expected predictors exist
missing_columns = [col for col in PREDICTORS if col not in df.columns]

print("Expected predictors:", len(PREDICTORS))
print("Available predictors:", len(PREDICTORS) - len(missing_columns))
print("Missing columns:", missing_columns)

# Build an audit table without changing the original dataset
predictor_audit = pd.DataFrame({
    "column": PREDICTORS,
    "dtype": [
        df[col].dtype if col in df.columns else None
        for col in PREDICTORS
    ],
    "missing_count": [
        df[col].isna().sum() if col in df.columns else None
        for col in PREDICTORS
    ],
    "missing_percentage": [
        round(df[col].isna().mean() * 100, 2)
        if col in df.columns else None
        for col in PREDICTORS
    ],
    "unique_values": [
        df[col].nunique(dropna=True) if col in df.columns else None
        for col in PREDICTORS
    ]
})

display(predictor_audit)

In [ ]:
# Inspect ranges and category values for the agreed predictors.

numeric_predictors = [
    "maternal_age",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "multiple",
    "anc_visits",
    "first_anc_month",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets"
]

categorical_predictors = [
    "education",
    "wealth_quintile",
    "residence"
]

print("=== NUMERIC PREDICTOR RANGES ===")

display(
    df[numeric_predictors]
    .describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99])
    .T
)

print("\n=== CATEGORICAL PREDICTOR VALUES ===")

for col in categorical_predictors:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).sort_index())

print("\n=== BINARY PREDICTOR VALUES ===")

binary_predictors = [
    "prior_stillbirth",
    "prior_loss",
    "multiple",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets"
]

for col in binary_predictors:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).sort_index())

In [ ]:
# AfyaNova: investigate potential data-quality issues
# Original df remains unchanged.

# 1. Inspect ANC visit counts
print("=== ANC VISITS: FREQUENCIES ===")
display(
    df["anc_visits"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("anc_visits")
    .to_frame("records")
)

# 2. Investigate zero values for first ANC month
print("\n=== FIRST ANC MONTH: FREQUENCIES ===")
display(
    df["first_anc_month"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("first_anc_month")
    .to_frame("records")
)

print("\nRecords where first_anc_month == 0:")
display(
    df.loc[
        df["first_anc_month"].eq(0),
        ["first_anc_month", "anc_visits", "perinatal_death"]
    ].head(20)
)

# 3. Inspect the largest preceding intervals
print("\n=== LARGEST PRECEDING INTERVALS ===")
display(
    df.loc[
        df["preceding_interval_m"].notna(),
        ["preceding_interval_m", "prior_live_births",
         "prior_stillbirth", "prior_loss"]
    ]
    .sort_values("preceding_interval_m", ascending=False)
    .head(20)
)

# 4. Summarise the upper tail of preceding intervals
print("\n=== PRECEDING INTERVAL QUANTILES ===")
display(
    df["preceding_interval_m"]
    .quantile([0.90, 0.95, 0.99, 0.995, 1.00])
    .rename("months")
    .to_frame()
)

# 5. Verify binary-variable coding, including missing values
binary_predictors = [
    "prior_stillbirth",
    "prior_loss",
    "multiple",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets"
]

print("\n=== BINARY VARIABLE VALUES ===")

for col in binary_predictors:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).sort_index())

# 6. Check whether any ANC measurements have unexpected negative values
anc_numeric = [
    "anc_visits",
    "first_anc_month"
]

print("\n=== NEGATIVE ANC VALUES ===")

for col in anc_numeric:
    negative_count = df[col].lt(0).sum()
    print(f"{col}: {negative_count} negative values")

In [ ]:
# AfyaNova — targeted data-quality checks
# This cell does not modify df.

# 1. Check the relationship between ANC visits and first ANC month
print("=== ANC VISITS vs FIRST ANC MONTH ===")

anc_consistency = pd.crosstab(
    df["anc_visits"],
    df["first_anc_month"],
    margins=True,
    dropna=False
)

display(anc_consistency)

# Check the specific zero-value patterns
no_anc = df["anc_visits"].eq(0)
no_first_month = df["first_anc_month"].eq(0)

print("\n=== ZERO-VALUE CONSISTENCY ===")
print("ANC visits = 0:", int(no_anc.sum()))
print("First ANC month = 0:", int(no_first_month.sum()))
print("Both equal 0:", int((no_anc & no_first_month).sum()))
print("ANC visits = 0 but first ANC month is not 0:",
      int((no_anc & ~no_first_month).sum()))
print("First ANC month = 0 but ANC visits are not 0:",
      int((no_first_month & ~no_anc).sum()))

# 2. Show unusual ANC records for review
print("\n=== ANC VISITS ABOVE 10 ===")

id_columns = [c for c in ["caseid", "pidx"] if c in df.columns]

review_columns = id_columns + [
    "anc_visits",
    "first_anc_month",
    "perinatal_death"
]

display(
    df.loc[df["anc_visits"] > 10, review_columns]
      .sort_values("anc_visits", ascending=False)
)

# 3. Review the largest preceding intervals
print("\n=== PRECEDING INTERVALS ABOVE 180 MONTHS ===")

interval_columns = id_columns + [
    "preceding_interval_m",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "maternal_age"
]

display(
    df.loc[df["preceding_interval_m"] > 180, interval_columns]
      .sort_values("preceding_interval_m", ascending=False)
)

# 4. Validate binary predictor values
print("\n=== BINARY CODING VALIDATION ===")

binary_predictors = [
    "prior_stillbirth",
    "prior_loss",
    "multiple",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets"
]

binary_results = []

for col in binary_predictors:
    observed = set(df[col].dropna().unique())
    unexpected = sorted(observed - {0, 1})

    binary_results.append({
        "variable": col,
        "observed_values": sorted(observed),
        "missing_count": int(df[col].isna().sum()),
        "unexpected_values": unexpected,
        "coding_valid": len(unexpected) == 0
    })

display(pd.DataFrame(binary_results))

# 5. Confirm the original dataset has not been changed by these checks
print("\nValidation checks completed. No values were modified.")

In [ ]:
# Investigate missing intervals among records reporting previous pregnancy history.
# This cell does not modify df.

interval_missing = df["preceding_interval_m"].isna()

has_previous_history = (
    df["prior_live_births"].gt(0)
    | df["prior_stillbirth"].eq(1)
    | df["prior_loss"].eq(1)
)

# Missing interval despite some reported pregnancy history
unexplained_missing = df.loc[
    interval_missing & has_previous_history,
    [
        "caseid",
        "pidx",
        "maternal_age",
        "prior_live_births",
        "prior_stillbirth",
        "prior_loss",
        "preceding_interval_m",
        "perinatal_death"
    ]
]

print("Missing intervals in total:", int(interval_missing.sum()))
print(
    "Missing intervals with no previous history:",
    int((interval_missing & ~has_previous_history).sum())
)
print(
    "Missing intervals with some previous history:",
    len(unexplained_missing)
)

display(unexplained_missing)

# Summarise the previous-history patterns in these records
print("\n=== HISTORY PATTERNS AMONG THESE RECORDS ===")

display(
    unexplained_missing.groupby(
        [
            "prior_live_births",
            "prior_stillbirth",
            "prior_loss"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

In [ ]:
# Examine interval missingness against previous pregnancy history

df_interval_audit = df[
    [
        "prior_live_births",
        "prior_stillbirth",
        "prior_loss",
        "preceding_interval_m"
    ]
].copy()

df_interval_audit["interval_missing"] = (
    df_interval_audit["preceding_interval_m"].isna()
)

df_interval_audit["no_previous_history"] = (
    df_interval_audit["prior_live_births"].eq(0)
    & df_interval_audit["prior_stillbirth"].eq(0)
    & df_interval_audit["prior_loss"].eq(0)
)

interval_missing_summary = (
    df_interval_audit
    .groupby("no_previous_history", dropna=False)
    .agg(
        records=("preceding_interval_m", "size"),
        missing_intervals=("interval_missing", "sum"),
        observed_intervals=("interval_missing", lambda x: (~x).sum())
    )
    .reset_index()
)

interval_missing_summary["missing_pct"] = (
    interval_missing_summary["missing_intervals"]
    / interval_missing_summary["records"]
    * 100
).round(2)

display(interval_missing_summary)

## Summary of Exploratory Data Analysis

- **Dataset structure:** 10,590 records and 57 columns. All 24 expected columns are available.
- **Target distribution:** 315 perinatal deaths (2.97%) and 10,275 non-deaths (97.03%), indicating class imbalance.
- **Duplicate checks:** No exact duplicate rows were found. Repeated `caseid` values require consideration when splitting data.
- **Missing values:** `preceding_interval_m` has 2,670 missing values. Of these, 2,631 occur among records without previous pregnancy history, while 39 occur despite previous pregnancy history.
- **Predictor validation:** All agreed predictors are available. Unusual ANC visit counts and long preceding intervals were flagged for review rather than automatically altered.
- **EDA boundary:** The original source dataframe `df` was not modified during EDA.

### EDA conclusion and transition to cleaning

The initial audit confirms that the dataset contains the required predictors and an internally consistent target. The following sections document the cleaning decisions, preserve an audit trail, validate the cleaned cohort, and export a standardized CSV for later harmonisation with the Tanzania cohort.


---

# Part II: Data Cleaning and Preparation

This part begins after the exploratory analysis. The original dataframe `df` is retained as the source record. All transformations are applied to `df_clean`, a separate working copy.


## 10. Create a protected working copy and define the modelling specification

The target is `perinatal_death`. The agreed model predictors comprise nine core maternal/history variables and six ANC variables. Survey weight and cluster are retained as helper fields, while identifiers are retained for quality assurance only.


In [ ]:
# Preserve the source dataframe and create a separate cleaning copy
df_raw = df.copy(deep=True)
df_clean = df_raw.copy(deep=True)

TARGET = TARGET_PERINATAL

CORE_PREDICTORS = [
    "maternal_age",
    "education",
    "wealth_quintile",
    "residence",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "multiple",
]

ANC_PREDICTORS = [
    "anc_visits",
    "first_anc_month",
    "bp_measured",
    "urine_tested",
    "blood_tested",
    "iron_tablets",
]

MODEL_PREDICTORS = CORE_PREDICTORS + ANC_PREDICTORS
HELPER_COLUMNS = ["weight", "v001"]
QA_COLUMNS = [col for col in ["caseid", "pidx"] if col in df_clean.columns]

required_columns = [TARGET] + MODEL_PREDICTORS + HELPER_COLUMNS
missing_required_columns = [
    col for col in required_columns if col not in df_clean.columns
]

print("Source shape:", df_raw.shape)
print("Working-copy shape:", df_clean.shape)
print("Required columns:", len(required_columns))
print("Missing required columns:", missing_required_columns)
print("Available QA columns:", QA_COLUMNS)

if missing_required_columns:
    raise ValueError(f"Required columns are missing: {missing_required_columns}")

assert df_raw.equals(df), "The raw copy should match the original EDA dataframe."


## 11. Baseline checks on the cleaning copy

Confirm the starting row count, exact duplicates, target distribution, and missingness in the agreed predictors. These checks provide a baseline against which the cleaning results can be compared.


In [ ]:
EXPECTED_ROWS = len(df_raw)

print("Rows at start:", EXPECTED_ROWS)
print("Columns at start:", df_clean.shape[1])
print("Exact duplicate rows:", df_clean.duplicated().sum())

print("\nTarget distribution:")
display(df_clean[TARGET].value_counts(dropna=False).sort_index().rename("records").to_frame())

print("\nTarget percentages:")
display(
    (df_clean[TARGET].value_counts(normalize=True, dropna=False)
     .sort_index().mul(100).round(2).rename("percentage").to_frame())
)

print("\nMissing values in agreed predictors:")
display(
    pd.DataFrame({
        "missing_count": df_clean[MODEL_PREDICTORS].isna().sum(),
        "missing_percentage": (df_clean[MODEL_PREDICTORS].isna().mean() * 100).round(2),
    }).sort_values("missing_count", ascending=False)
)


## 12. Validate the target

The target must be present and contain only the binary values 0 and 1. The target distribution is expected to remain unchanged throughout cleaning.


In [ ]:
observed_target_values = set(df_clean[TARGET].dropna().unique())

print("Observed target values:", observed_target_values)
print("Missing target values:", df_clean[TARGET].isna().sum())

assert df_clean[TARGET].notna().all(), "The target contains missing values."
assert observed_target_values.issubset({0, 1}), "Unexpected target coding."

target_counts_before_cleaning = df_clean[TARGET].value_counts().sort_index().to_dict()
print("Target validation passed.")


## 13. Classify missing preceding intervals

A missing preceding interval is structural when the record has no previous live births, stillbirths, or pregnancy losses. Such values must remain missing; they must not be replaced with zero.

Missing intervals among records with previous pregnancy history are classified separately for investigation.


In [ ]:
history_columns = [
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
]

history_complete = df_clean[history_columns].notna().all(axis=1)

no_previous_history = (
    history_complete
    & df_clean[history_columns].eq(0).all(axis=1)
)

has_previous_history = (
    history_complete
    & df_clean[history_columns].gt(0).any(axis=1)
)

interval_missing = df_clean["preceding_interval_m"].isna()

df_clean["preceding_interval_status"] = np.select(
    [
        ~interval_missing,
        interval_missing & no_previous_history,
        interval_missing & has_previous_history,
        interval_missing & ~history_complete,
    ],
    [
        "Available",
        "No previous pregnancy history",
        "Missing despite pregnancy history",
        "History status unknown",
    ],
    default="Review required",
)

df_clean["preceding_interval_missing"] = interval_missing.astype("int8")
df_clean["no_previous_pregnancy_history"] = no_previous_history.astype("int8")

print("Preceding interval status:")
display(
    df_clean["preceding_interval_status"]
    .value_counts(dropna=False)
    .rename("records")
    .to_frame()
)

print("\nCross-tabulation of status and interval missingness:")
display(pd.crosstab(
    df_clean["preceding_interval_status"],
    df_clean["preceding_interval_missing"],
    margins=True,
))


## 14. Verify the missing-interval classification against the EDA audit

The EDA identified 2,631 structural missing intervals, 39 missing intervals despite pregnancy history, and 7,920 observed intervals. The classification must reproduce these counts before imputation is permitted.


In [ ]:
expected_interval_counts = {
    "No previous pregnancy history": 2631,
    "Missing despite pregnancy history": 39,
    "Available": 7920,
}

observed_interval_counts = (
    df_clean["preceding_interval_status"].value_counts().to_dict()
)

classification_check = pd.DataFrame({
    "expected_from_eda": pd.Series(expected_interval_counts),
    "observed_now": pd.Series(observed_interval_counts),
})
classification_check["matches"] = (
    classification_check["expected_from_eda"]
    == classification_check["observed_now"]
)

display(classification_check)

unknown_or_review = df_clean["preceding_interval_status"].isin(
    ["History status unknown", "Review required"]
).sum()

print("Missing-history cases requiring review:", unknown_or_review)

assert classification_check["matches"].all(), (
    "Interval classification does not match the EDA audit."
)
assert unknown_or_review == 0, "Some records require additional history review."

print("Missing-interval classification matches the EDA audit.")


## 15. Inspect the 39 unexplained missing intervals

The 39 records are reviewed using the previous pregnancy-history fields. This is an investigation step; no values are changed in this section.


In [ ]:
review_records = df_clean.loc[
    df_clean["preceding_interval_status"]
    == "Missing despite pregnancy history"
].copy()

print("Records requiring review:", len(review_records))

display(
    review_records[
        history_columns + ["preceding_interval_m"]
    ]
    .value_counts(dropna=False)
    .reset_index(name="records")
    .sort_values("records", ascending=False)
)

display(review_records[history_columns].describe())


### Review finding

The EDA and cleaning review identified:
- 37 records with one previous live birth, no previous stillbirth, and no previous pregnancy loss.
- 2 records with no previous live births, one previous stillbirth, and one previous pregnancy loss.

These 39 records have a previous pregnancy history but no recorded preceding interval. They are therefore distinguished from the 2,631 structural missing values.


## 16. Impute only the 39 unexplained missing intervals

For this country-specific cleaning export, the 39 unexplained missing intervals are imputed using the median of observed Kenya intervals. The 2,631 structural missing intervals are deliberately retained as missing.

**Modelling caveat:** this descriptive cleaning export records the audit decision. In final model evaluation, the imputation statistic must be learned from the training partition only and then applied to validation/test data to avoid leakage.


In [ ]:
# Preserve original interval values for the audit trail
df_clean["preceding_interval_m_original"] = (
    df_clean["preceding_interval_m"].copy()
)

# Select only unexplained missing intervals
imputation_mask = (
    df_clean["preceding_interval_status"]
    == "Missing despite pregnancy history"
)

# Calculate the median from observed Kenya intervals
interval_median = df_clean.loc[
    df_clean["preceding_interval_m"].notna(),
    "preceding_interval_m"
].median()

if pd.isna(interval_median):
    raise ValueError("Cannot impute: no observed preceding intervals are available.")

# Impute only the 39 unexplained missing values
df_clean.loc[imputation_mask, "preceding_interval_m"] = interval_median

# Record which values were imputed
df_clean["preceding_interval_imputed"] = imputation_mask.astype("int8")

print("Median preceding interval:", interval_median)
print("Records imputed:", int(imputation_mask.sum()))
print("\nMissing values after imputation:")
print(df_clean["preceding_interval_m"].isna().sum())
print("\nImputation indicator:")
display(df_clean["preceding_interval_imputed"].value_counts().rename("records").to_frame())


## 17. Validate the imputation audit

The validation confirms that exactly 39 records were imputed, original observed values were not changed, and all structural missing intervals remain missing.


In [ ]:
# 1. Confirm the original dataset size is preserved
assert len(df_clean) == EXPECTED_ROWS == 10590

# 2. Confirm exactly 39 records were imputed
assert int(df_clean["preceding_interval_imputed"].sum()) == 39

# 3. Confirm all imputed records now contain the median
imputed_mask = df_clean["preceding_interval_imputed"].eq(1)
imputed_values = df_clean.loc[imputed_mask, "preceding_interval_m"]

assert imputed_values.notna().all()
assert imputed_values.eq(interval_median).all()

# 4. Confirm originally observed values remain unchanged
originally_observed = df_clean["preceding_interval_m_original"].notna()
assert df_clean.loc[originally_observed, "preceding_interval_m"].equals(
    df_clean.loc[originally_observed, "preceding_interval_m_original"]
)

# 5. Confirm structural missing values remain missing
structural_missing = df_clean["preceding_interval_status"].eq(
    "No previous pregnancy history"
)
assert int(structural_missing.sum()) == 2631
assert df_clean.loc[structural_missing, "preceding_interval_m"].isna().all()

# 6. Confirm no unexplained missing intervals remain
unexplained_missing = df_clean["preceding_interval_status"].eq(
    "Missing despite pregnancy history"
)
assert df_clean.loc[unexplained_missing, "preceding_interval_m"].notna().all()

print("PRECEDING INTERVAL IMPUTATION AUDIT")
print("-----------------------------------")
print("Dataset rows:", len(df_clean))
print("Median used:", interval_median)
print("Records imputed:", int(imputed_mask.sum()))
print("Original observed values preserved:", int(originally_observed.sum()))
print("Structural missing values retained:", int(structural_missing.sum()))
print(
    "Remaining unexplained missing values:",
    int(df_clean.loc[unexplained_missing, "preceding_interval_m"].isna().sum())
)
print("\nAll imputation validation checks passed.")


## 18. Prepare the cleaned export

The export retains the target, 15 agreed predictors, audit fields, survey helpers, available QA identifiers, and a country identifier. The original dataframe `df` remains unchanged. The audit fields and helper columns are retained for traceability and downstream analysis.

In [ ]:
# Identify audit fields
AUDIT_COLUMNS = [
    "preceding_interval_status",
    "preceding_interval_missing",
    "no_previous_pregnancy_history",
    "preceding_interval_imputed",
    "preceding_interval_m_original",
]

# Add country identifier
df_clean["country"] = "Kenya"

# Retain QA fields only when available
AVAILABLE_QA_COLUMNS = [
    col for col in ["caseid", "pidx"] if col in df_clean.columns
]

EXPORT_COLUMNS = (
    [TARGET]
    + MODEL_PREDICTORS
    + AUDIT_COLUMNS
    + HELPER_COLUMNS
    + AVAILABLE_QA_COLUMNS
    + ["country"]
)

# Remove accidental duplicate names while preserving order
EXPORT_COLUMNS = list(dict.fromkeys(EXPORT_COLUMNS))

missing_export_columns = [
    col for col in EXPORT_COLUMNS if col not in df_clean.columns
]
if missing_export_columns:
    raise ValueError(f"Missing export columns: {missing_export_columns}")

df_export = df_clean[EXPORT_COLUMNS].copy()

print("Export dataset shape:", df_export.shape)
print("\nExport columns:")
print(df_export.columns.tolist())
display(df_export.head())


## 19. Validate the export dataset

Check the export shape, unique column names, target integrity, country label, and interval audit. The expected export contains 10,590 rows and 26 columns.


In [ ]:
assert len(df_export) == EXPECTED_ROWS
assert df_export.shape == (10590, 26)
assert df_export.columns.is_unique

assert df_export[TARGET].notna().all()
assert set(df_export[TARGET].unique()).issubset({0, 1})
assert df_export["country"].eq("Kenya").all()

assert int(df_export["preceding_interval_imputed"].sum()) == 39
assert df_export.loc[
    df_export["no_previous_pregnancy_history"].eq(1),
    "preceding_interval_m"
].isna().all()

assert df_export.loc[
    df_export["preceding_interval_status"].eq(
        "Missing despite pregnancy history"
    ),
    "preceding_interval_m"
].notna().all()

assert all(col in df_export.columns for col in MODEL_PREDICTORS)

print("EXPORT VALIDATION PASSED")
print("------------------------")
print("Rows:", len(df_export))
print("Columns:", len(df_export.columns))
print("Model predictors:", len(MODEL_PREDICTORS))
print("Target:", TARGET)
print("Imputed intervals:", int(df_export["preceding_interval_imputed"].sum()))
print("Country:", df_export["country"].unique())
print("\nAll export validation checks passed.")


## 20. Export and verify the cleaned CSV

The cleaned CSV is written to `data/processed/ke_perinatal_anc_clean.csv`. The file is reloaded after export to verify that its shape and column order match the in-memory export dataframe.


In [ ]:
output_path = repo_root / "data" / "processed" / "ke_perinatal_anc_clean.csv"

df_export.to_csv(output_path, index=False)

print("Cleaned dataset successfully exported!")
print("File:", output_path)
print("Shape:", df_export.shape)

# Reload the saved CSV and verify it
df_check = pd.read_csv(output_path)

assert df_check.shape == df_export.shape
assert df_check.columns.tolist() == df_export.columns.tolist()

print("\nSAVED FILE VERIFICATION PASSED")
print("------------------------------")
print("Saved dataset shape:", df_check.shape)
print("Saved dataset size:", round(output_path.stat().st_size / 1024, 2), "KB")
print("\nMissing values in preceding_interval_m:")
print(df_check["preceding_interval_m"].isna().sum())
print("\nTarget distribution:")
display(df_check[TARGET].value_counts().sort_index().rename("records").to_frame())
print("\nCountry distribution:")
display(df_check["country"].value_counts().to_frame())


## 21. Cleaning summary and modelling handover

### Cleaning outcomes

The Kenya perinatal ANC cohort contains 10,590 records. The preceding-interval audit identified 7,920 observed values, 2,631 structural missing values among records with no previous pregnancy history, and 39 unexplained missing values. The 39 unexplained values were imputed using the observed Kenya median of 36 months. Structural missing values were retained.

The cleaned export contains 10,590 rows and 26 columns. The target distribution remains 10,275 non-deaths (97.03%) and 315 perinatal deaths (2.97%).

### Column-use guidance

| Column group | Fields | Intended use |
|---|---|---|
| Target | `perinatal_death` | Outcome label |
| Model predictors | 9 core + 6 ANC variables | Candidate model inputs |
| Survey helpers | `weight`, `v001` | Survey-weighted analysis and cluster-aware splitting |
| QA identifiers | `caseid`, `pidx` | Record traceability and grouping checks |
| Audit fields | Interval status, missingness flags, original interval, imputation flag | Cleaning transparency and quality assurance |
| Country | `country` | Country identification and later harmonisation |

We will select `MODEL_PREDICTORS` explicitly and keep `TARGET` separate. IDs, survey helpers, audit fields, and country should be handled according to the modelling design rather than included automatically.

### Modelling considerations

- The perinatal-death target is highly imbalanced (2.97% positive). Accuracy alone will not be sufficient; evaluate recall, precision, F1, PR-AUC, ROC-AUC, balanced accuracy, and the confusion matrix.
- Use cluster-aware splitting based on `v001`, and consider repeated `caseid` values when designing the split to reduce leakage.
- Fit imputation and other learned preprocessing steps on the training data only, then apply them to validation/test data. The full-cohort median of 36 months is recorded here for the country-specific cleaning audit, not as a value to hard-code into leakage-sensitive model evaluation.

### Output

`data/processed/ke_perinatal_anc_clean.csv`

This notebook contains the EDA and cleaning record for the Kenya perinatal ANC cohort. The original source CSV remains unchanged.
